In [1]:
import os
os.chdir("/Users/cherie/Desktop/quantum-gates/src")  # or any folder that actually exists
print(os.getcwd())

from quantum_gates._utility.RotatedSurfaceCodeLoom import RotatedSurfaceCodeLoom
import matplotlib.pyplot as plt
import numpy as np

/Users/cherie/Desktop/quantum-gates/src


In [10]:
code = RotatedSurfaceCodeLoom(distance = 3, n_cycles = 5, n_shots = 100, noise = True, p = 0.06)

In [11]:
logical_error_rate = code.run_circ("MrAnderson")
print(f"Logical error rate: {logical_error_rate:.4f}")

n_qubits: 17
Circuit transpiled
Simulation complete
MrAnderson result: {'0001000011010011100010100111011101101101011000000': 1, '0000000000011000010000011100101000100000010000001': 1, '1111111111111111111111111111111111111111111111111': 29, '1111111111110000101000001110111100010001000000000': 1, '0000111110110110010010100010010011000110000001011': 1, '0011100100001111111101101111111011010101011000001': 1, '0000001000100010011010001000000010010011000001000': 1, '0011011100011000011000000000010000001000000000001': 1, '0011111110001010011010000110111011001100111000110': 1, '0001101100100101101000011000010000110111100000011': 1, '0000011111010001001000011100101111110010010011111': 1, '0000010010010110000000010110001000111001001111111': 1, '0001101111111100010000110000010101100100001100001': 1, '1111111110010010101100100000010001111001101101001': 1, '0100011110001111100110111000110110010000111110010': 1, '0000010100111111111011010100011000111001100001110': 1, '111111101000001100111001010100

In [5]:
def plot_results(results_3, results_5, results_7):
    p_values_3  = [r[0] for r in results_3]
    ler_values_3 = [r[1] for r in results_3]
    p_values_5  = [r[0] for r in results_5]
    ler_values_5 = [r[1] for r in results_5]
    p_values_7  = [r[0] for r in results_7]
    ler_values_7 = [r[1] for r in results_7]

    # Use all p_values for the physical error rate line
    all_p = sorted(set(p_values_3 + p_values_5 + p_values_7))

    plt.figure(figsize=(8, 6))
    plt.plot(p_values_3, ler_values_3, 'bo-', label='d=3', linewidth=2, markersize=6)
    plt.plot(p_values_5, ler_values_5, 'gs-', label='d=5', linewidth=2, markersize=6)
    plt.plot(p_values_7, ler_values_7, 'r^-', label='d=7', linewidth=2, markersize=6)
    plt.plot(all_p, all_p, 'k--', label='Physical error rate (LER = p)', linewidth=2)

    plt.xlabel('Physical error rate p', fontsize=13)
    plt.ylabel('Logical error rate', fontsize=13)
    plt.title('Rotated Surface Code — Logical vs Physical Error Rate', fontsize=13)
    plt.xscale('log')
    plt.yscale('log')
    plt.legend(fontsize=12)
    plt.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    plt.savefig('surface_code_ler.png', dpi=150)
    plt.show()
    print("Plot saved to surface_code_ler.png")

In [ ]:
results_3 = []
results_5 = []
results_7 = []
p_values = np.logspace(-4, -1, 20)
for p in p_values:
    code = RotatedSurfaceCodeLoom(distance = 3, n_cycles = 5, n_shots = 10000, noise = True, p = p)
    ler = code.run_circ("AER")
    results_3.append((p, ler))
    print("Distance 3:")
    print(f"p={p:.5f} → LER={ler:.5f}  {'SUPPRESSED ✓' if ler < p else 'WORSE THAN PHYSICAL ✗'}")

    code = RotatedSurfaceCodeLoom(distance = 5, n_cycles = 5, n_shots = 10000, noise = True, p = p)
    ler = code.run_circ("AER")
    results_5.append((p, ler))
    print("Distance 5:")
    print(f"p={p:.5f} → LER={ler:.5f}  {'SUPPRESSED ✓' if ler < p else 'WORSE THAN PHYSICAL ✗'}")

    code = RotatedSurfaceCodeLoom(distance = 7, n_cycles = 5, n_shots = 10000, noise = True, p = p)
    ler = code.run_circ("AER")
    results_7.append((p, ler))
    print("Distance 7:")
    print(f"p={p:.5f} → LER={ler:.5f}  {'SUPPRESSED ✓' if ler < p else 'WORSE THAN PHYSICAL ✗'}")



In [ ]:
plot_results(results_3, results_5, results_7)

In [ ]:
error_rates = []
for i in range(100):
    logical_error_rate = code.run_circ("MrAnderson")
    error_rates.append(logical_error_rate)
mean_error_rate = sum(error_rates) / len(error_rates)
print("Mean logical error rate:", mean_error_rate)


"Statevector collapsed to zero norm after circuit execution" - bug?

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister


# 7 qubits, 8 classical bits (indices 0–7)
qr = QuantumRegister(7, 'q')
cr = ClassicalRegister(7, 'c')
qc = QuantumCircuit(qr, cr)

# --- Initial Hadamards ---
qc.h(qr[2])
qc.h(qr[5])
qc.h(qr[6])

# --- First layer of CNOTs ---
# q2 (control) -> q0 (target)
qc.cx(qr[2], qr[0])
# q5 (control) -> q3 (target)
qc.cz(qr[5], qr[3])

# q6 (control) -> q1 (target)  (based on line routing)
qc.cz(qr[5], qr[0])

# --- Second layer of CNOTs ---
# q0 (control) -> q1 (target)
qc.cx(qr[2], qr[1])
# q2 (control) -> q3 (target)
qc.cx(qr[6], qr[3])

# --- Second Hadamards (mid-circuit) ---
qc.h(qr[2])
qc.cz(qr[4], qr[5])  # q5 (control) -> q0 (target)
# --- Measurements (first wave: bits 0, 3) ---
qc.measure(qr[0], cr[0])
qc.measure(qr[3], cr[3])

# --- Third CNOT layer ---
# q1 (control) -> q4 (target)
qc.cz(qr[1], qr[5])
# q6 (control) -> q4 (target)
qc.cx(qr[6], qr[4])

# --- Measurements (second wave: bit 2) ---
qc.measure(qr[2], cr[2])

# --- Final Hadamards ---
qc.h(qr[5])
qc.h(qr[6])

# --- Final measurements (bits 1, 4, 5, 6) ---
qc.measure(qr[1], cr[1])
qc.measure(qr[4], cr[4])
qc.measure(qr[5], cr[5])
qc.measure(qr[6], cr[6])
qc.barrier()
qc.x(qr[0])
qc.x(qr[1])
qc.x(qr[2])
qc.x(qr[3])
qc.x(qr[4])
qc.x(qr[5])
qc.x(qr[6])
qc.draw('mpl')

In [10]:
from qiskit import transpile
from qiskit.circuit.controlflow import ControlFlowOp 
from qiskit.transpiler import CouplingMap
from quantum_gates.utilities import DeviceParameters


def _transpile_circ(qc, n_qubits):
        """
        This function transpiles the input circuit using a fixed basis gate set
        and a nearest-neighbor coupling map. It also detects whether the circuit
        contains control-flow operations and adapts the transpilation scheduling
        accordingly.
        """

        needs_controlflow = any(isinstance(op.operation, ControlFlowOp) for op in qc.data)

        cm = CouplingMap([(i, i+1) for i in range(n_qubits - 1)])

        t_circ = transpile(
            qc,
            None,
            basis_gates=['ecr', 'rz', 'sx','x'],
            coupling_map=cm,
            initial_layout=list(range(n_qubits)),
            seed_transpiler=42,
            scheduling_method=needs_controlflow,
        )
        return t_circ

def _get_device_parameters(backend, n_qubits):
        """
        This method loads single- and two-qubit noise parameters from a given
        backend into a DeviceParameters object and converts them into a lookup
        dictionary suitable for simulation.
        """

        device_param = DeviceParameters(list(range(n_qubits)))
        device_param.load_from_backend(backend)
        device_param_lookup = device_param.__dict__()
        # The standard t_int value for allowed ECR gates
        t_int_value = 6.6e-07
        #Add t_int values for all consecutive qubit pairs
        non_zero_p_int =  device_param_lookup['p_int'][ device_param_lookup['p_int'] != 0]
        # Get average p_int for fallback
        avg_p_int = np.mean(non_zero_p_int) if len(non_zero_p_int) > 0 else 0.01

        for i in range(n_qubits - 1):
            # Add both directions for the ECR gate
            device_param_lookup['t_int'][i, i+1] = t_int_value
            device_param_lookup['t_int'][i+1, i] = t_int_value

            if device_param_lookup['p_int'][i, i+1] == 0:
                # Look for nearest existing edge with p_int value
                found_p_int = None
                
                # Check adjacent qubits first
                for offset in [1, 2, 3]:
                    if i - offset >= 0 and device_param_lookup['p_int'][i-offset, i-offset+1] != 0:
                        found_p_int = device_param_lookup['p_int'][i-offset, i-offset+1]
                        break
                    if i + offset < n_qubits - 1 and device_param_lookup['p_int'][i+offset, i+offset+1] != 0:
                        found_p_int = device_param_lookup['p_int'][i+offset, i+offset+1]
                        break
                
                # Fall back to average if no nearby edge found
                if found_p_int is None:
                    found_p_int = avg_p_int
                
                device_param_lookup['p_int'][i, i+1] = found_p_int
                device_param_lookup['p_int'][i+1, i] = found_p_int
        return device_param_lookup  

In [ ]:
from quantum_gates.simulators import MrAndersonSimulator
from quantum_gates.gates import standard_gates, noise_free_gates
from quantum_gates.circuits import EfficientCircuit
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

noise = True
bit_flip_bool = True
n_qubits = 7
n_shots = 100

if noise:
    set_gate = standard_gates
else:
    set_gate =  noise_free_gates
sim = MrAndersonSimulator(gates=set_gate, CircuitClass=EfficientCircuit)

initial_psi = np.zeros(2**n_qubits)
initial_psi[0] = 1.0  # set |00...0⟩

backend = FakeBrisbane() 
device_param_lookup = _get_device_parameters(backend, n_qubits)
t_circ = _transpile_circ(qc, n_qubits)

# Run simulation
res  = sim.run( 
    t_qiskit_circ=t_circ, 
    psi0=initial_psi, 
    shots=n_shots, 
    device_param=device_param_lookup,
    nqubit=n_qubits,
    bit_flip_bool=bit_flip_bool,
    )

print("Results:", res["mid_counts"])